# RCAug — Aumento de dados fiel ao mestrado (André Dias)
Rodar no **Google Colab** (como ele fez). Reproduz as 6 transformações da execução final dele (`horizontal_flip`, `vertical_flip`, `rotation`, `transpose`, `grid_distortion`, `color_transfer`) com o **color_transfer exato** dele (transferência de Reinhard no espaço LAB restrita aos núcleos).

**Correção necessária** vs. o script anexo: aplica as transformações geométricas **também na máscara real** e **salva a máscara** — o notebook original usava máscara dummy e salvava só a imagem, o que seria inválido para treino de segmentação.

Só aumenta o **treino**. Val e test ficam intactos.

In [ ]:
# CÉLULA 1 — Instalar dependências
# Sem openslide/torch: só eram usados no inpainting/GAN, que o André NÃO usou na execução final.
!pip install -q albumentations scikit-image pillow numpy

In [ ]:
# CÉLULA 2 — Montar o Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# CÉLULA 3 — Imports e funções do pipeline do André (fiéis ao Aumento_Dali.ipynb)
import os, random
import numpy as np
from PIL import Image
import skimage.color as sk_color
from albumentations import RandomRotate90, Transpose, GridDistortion

def rgb_to_lab(np_img):
    if np_img.ndim == 3 and np_img.shape[2] > 3:
        np_img = sk_color.rgba2rgb(np_img)
    lab = sk_color.rgb2lab(np_img)
    lab = ((lab + [0, 128, 128]) / [100, 255, 255])   # normaliza p/ [0,1]
    return lab

def lab_to_rgb(np_img):
    lab_rescaled = ((np_img - [0, 128, 128]) * [100, 255, 255]) / 255
    return sk_color.lab2rgb(lab_rescaled)

# color_transfer EXATO do André (Reinhard restrito a núcleos: L < 0.86)
def transfer_color(o, t, L_threshold=0.86):
    o_l = o[:, :, 0][(o[:, :, 0] < L_threshold)].mean()
    o_a = o[:, :, 1][(o[:, :, 0] < L_threshold)].mean()
    o_b = o[:, :, 2][(o[:, :, 0] < L_threshold)].mean()
    t_l = t[:, :, 0][(t[:, :, 0] < L_threshold)].mean()
    t_a = t[:, :, 1][(t[:, :, 0] < L_threshold)].mean()
    t_b = t[:, :, 2][(t[:, :, 0] < L_threshold)].mean()
    aug = np.copy(o); m = (o[:, :, 0] < L_threshold)
    aug[:, :, 0][m] = aug[:, :, 0][m] - o_l + t_l
    aug[:, :, 1][m] = aug[:, :, 1][m] - o_a + t_a
    aug[:, :, 2][m] = aug[:, :, 2][m] - o_b + t_b
    return aug

def load_rgb(path):
    with open(path, 'rb') as f:
        return Image.open(f).convert('RGB')

def load_mask(path):
    with open(path, 'rb') as f:
        return Image.open(f).convert('L')

In [ ]:
# CÉLULA 4 — Aumento de imagem + MÁSCARA juntas (fiel + correção)
# 6 transformações, cada uma ~50% de chance, independentes.
# Geométricas (flip/rotation/transpose/grid) -> imagem E máscara.
# color_transfer -> só imagem (não muda geometria, máscara intacta).
def augment_pair(image_pil, mask_pil, target_pil, aug_list):
    image = np.array(image_pil); mask = np.array(mask_pil); used = []

    if "horizontal_flip" in aug_list and random.random() > 0.5:
        image = np.ascontiguousarray(image[:, ::-1, :])
        mask  = np.ascontiguousarray(mask[:, ::-1]); used.append("horizontal_flip")

    if "vertical_flip" in aug_list and random.random() > 0.5:
        image = np.ascontiguousarray(image[::-1, :, :])
        mask  = np.ascontiguousarray(mask[::-1, :]); used.append("vertical_flip")

    if "rotation" in aug_list and random.random() > 0.5:
        a = RandomRotate90(p=1)(image=image, mask=mask)
        image, mask = a['image'], a['mask']; used.append("rotation")

    if "transpose" in aug_list and random.random() > 0.5:
        a = Transpose(p=1)(image=image, mask=mask)
        image, mask = a['image'], a['mask']; used.append("transpose")

    if "grid_distortion" in aug_list and random.random() > 0.5:
        a = GridDistortion(p=1)(image=image, mask=mask)
        image, mask = a['image'], a['mask']; used.append("grid_distortion")

    if "color_transfer" in aug_list and target_pil is not None and random.random() > 0.5:
        orig_lab = rgb_to_lab(image)
        tgt_lab  = rgb_to_lab(np.array(target_pil))
        rgb = lab_to_rgb(transfer_color(orig_lab, tgt_lab))
        image = (np.clip(rgb, 0, 1) * 255).astype('uint8'); used.append("color_transfer")

    mask_bin = (mask > 0).astype('uint8') * 255
    return Image.fromarray(image), Image.fromarray(mask_bin), used

In [ ]:
# CÉLULA 5 — Loop principal (AJUSTE os caminhos do SEU Drive)
INPUT_IMG_DIR  = "/content/drive/MyDrive/TCC/crop_in/train"
INPUT_MASK_DIR = "/content/drive/MyDrive/TCC/crop_in/train/mascaras"
OUT_IMG_DIR    = "/content/drive/MyDrive/TCC/crop_out/train"
OUT_MASK_DIR   = "/content/drive/MyDrive/TCC/crop_out/train/mascaras"

N_AUG = 3   # 638*(1+3) ≈ 2552, ≈ o do André
AUG = ["horizontal_flip","vertical_flip","rotation","transpose","grid_distortion","color_transfer"]

os.makedirs(OUT_IMG_DIR, exist_ok=True)
os.makedirs(OUT_MASK_DIR, exist_ok=True)

img_files = sorted([f for f in os.listdir(INPUT_IMG_DIR)
                    if f.lower().endswith(('.png','.jpg','.jpeg','.tif'))])
print("Crops de treino encontrados:", len(img_files))

copied = made = 0
for img_name in img_files:
    mask_path = os.path.join(INPUT_MASK_DIR, img_name)
    if not os.path.exists(mask_path):
        print("  [aviso] sem máscara para", img_name, "- pulando"); continue
    image_pil = load_rgb(os.path.join(INPUT_IMG_DIR, img_name))
    mask_pil  = load_mask(mask_path)

    # copia o ORIGINAL (mantém os crops sem aumento no conjunto)
    image_pil.save(os.path.join(OUT_IMG_DIR, img_name))
    Image.fromarray((np.array(mask_pil) > 0).astype('uint8')*255).save(os.path.join(OUT_MASK_DIR, img_name))
    copied += 1

    for k in range(N_AUG):
        target_pil = load_rgb(os.path.join(INPUT_IMG_DIR, random.choice(img_files)))
        aug_img, aug_mask, _ = augment_pair(image_pil, mask_pil, target_pil, AUG)
        stem, ext = os.path.splitext(img_name)
        out_name = f"{stem}_aug{k}{ext}"
        aug_img.save(os.path.join(OUT_IMG_DIR, out_name))
        aug_mask.save(os.path.join(OUT_MASK_DIR, out_name))
        made += 1

print("Originais copiados:", copied)
print("Aumentados gerados:", made)
print("TOTAL treino:", copied + made)

In [ ]:
# CÉLULA 6 — Conferência visual (confirme imagem<->máscara batendo)
import matplotlib.pyplot as plt
sample = sorted(os.listdir(OUT_IMG_DIR))[5]
im = Image.open(os.path.join(OUT_IMG_DIR, sample))
mk = Image.open(os.path.join(OUT_MASK_DIR, sample))
fig, ax = plt.subplots(1, 3, figsize=(10,4))
ax[0].imshow(im);              ax[0].set_title(sample);    ax[0].axis('off')
ax[1].imshow(mk, cmap='gray'); ax[1].set_title('máscara'); ax[1].axis('off')
ax[2].imshow(im); ax[2].imshow(mk, cmap='jet', alpha=0.4); ax[2].set_title('overlay'); ax[2].axis('off')
plt.tight_layout(); plt.show()